# Psychological-Bias-Transfer — Local Pilot
Run cells top-to-bottom. This is the local-friendly variant. Secrets come from `os.environ` or a `.env` in the repo root.

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ.get("PBT_REPO", "/home/forge/Obsidian Vault/Projects/Psychological-Bias-Transfer")).resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("REPO=", REPO)

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata
    _in_colab = True
except ImportError:
    userdata = None
    _in_colab = False

def _get(key):
    if _in_colab:
        try:
            return userdata.get(key)
        except Exception:
            return None
    val = os.environ.get(key)
    if val:
        return val
    env_path = ".env"
    if os.path.exists(env_path):
        for line in open(env_path):
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() == key:
                return v.strip().strip('"').strip("'")
    return None

HF_TOKEN = _get("HF_TOKEN")
MINIMAX_API_KEY = _get("MINIMAX_API_KEY")
JUDGE_BACKEND = _get("JUDGE_BACKEND") or "minimax"
JUDGE_MODEL = _get("JUDGE_MODEL") or "minimax-m3"

os.environ["HF_TOKEN"] = HF_TOKEN or ""
os.environ["MINIMAX_API_KEY"] = MINIMAX_API_KEY or ""
os.environ["JUDGE_BACKEND"] = JUDGE_BACKEND
os.environ["JUDGE_MODEL"] = JUDGE_MODEL
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not _in_colab:
    print("Running locally. Secrets come from os.environ or .env.")
    print("  HF_TOKEN set?", bool(HF_TOKEN))
    print("  MINIMAX_API_KEY set?", bool(MINIMAX_API_KEY))

In [ ]:
!python3 -c "import importlib.util; import sys; sys.exit(0 if importlib.util.find_spec('datasketch') else 1)" || pip install datasketch==1.6.5
!python3 -c "import importlib.util; import sys; sys.exit(0 if importlib.util.find_spec('sklearn') else 1)" || pip install scikit-learn>=1.6.0

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print('HF_TOKEN is not set; dataset downloads may be rate-limited.')

In [ ]:
!python3 data/corpus_builder.py \
  --source hf_csv_source \
  --hf-dataset-id solomonk/reddit_mental_health_posts \
  --hf-configs depression.csv ptsd.csv aspergers.csv adhd.csv \
  --text-fields title body \
  --treatment-subreddits depression ptsd \
  --control-candidates aspergers adhd \
  --out-dir data/processed \
  --target-tokens 200000

In [ ]:
!python3 data/corpus_validator.py \
  --treatment data/processed/treatment_corpus.jsonl \
  --control data/processed/control_corpus.jsonl \
  --out-dir data/validation

In [ ]:
!python3 training/finetune_qlora.py --model qwen2.5-7b --corpus treatment --seed 42 --config config/training_config.yaml

In [ ]:
!python3 evaluation/generate_outputs.py \
  --base-model Qwen/Qwen2.5-7B-Instruct \
  --adapter checkpoints/treatment_seed42/final_adapter \
  --condition-name treatment_seed42 \
  --out data/generations/treatment_seed42.jsonl

In [ ]:
!python3 evaluation/judge.py \
  --generations data/generations/treatment_seed42.jsonl \
  --judge "$JUDGE_BACKEND" --model "$JUDGE_MODEL" \
  --out data/judged/treatment_seed42.jsonl

In [ ]:
!python3 analysis/statistical_analysis.py \
  --judged-dir data/judged --out-dir results \
  --corpus-hit-rate-report data/validation/validation_report.json

In [ ]:
!python3 scripts/build_release_artifacts.py \
  --processed-dir data/processed \
  --validation-dir data/validation \
  --results-dir results \
  --out-dir data/release